In [ ]:
# Importar pandas para leer el CSV y manipular los datos en forma de tabla.
import pandas as pd

# Importar las funciones de fastavro necesarias para generar el archivo Avro.
# writer escribe los registros y parse_schema prepara y valida el esquema.
from fastavro import writer, parse_schema


# read_csv carga el contenido del fichero en un DataFrame de pandas.
df = pd.read_csv(
    "TABLA_ACCIDENTES_24.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)


# Así se preparan los datos para los campos que admiten valores nulos.
df = df.where(pd.notnull(df), None)


# La lista fields almacenará la definición de cada columna del CSV.
# Cada campo incluirá su nombre, su tipo de dato y un valor predeterminado.
fields = []

# Recorrer todas las columnas del DataFrame para definir el esquema.
for columna in df.columns:

    # Comprobar si pandas identifica la columna como un tipo entero.
    if pd.api.types.is_integer_dtype(df[columna]):
        tipo = "long"

    elif pd.api.types.is_float_dtype(df[columna]):
        tipo = "double"

    # Las columnas que no son enteras ni decimales se definen como texto.
    else:
        tipo = "string"

    # Añadir la definición de la columna a la lista de campos del esquema.
    # "name" conserva el nombre original de la columna.
    fields.append({
        "name": columna,
        "type": ["null", tipo],
        "default": None
    })


# type="record" indica que cada registro contiene varios campos.
# name asigna un nombre al tipo de registro que representa los accidentes.
# namespace establece un espacio de nombres para identificar el esquema.
# fields incorpora las columnas generadas en el bucle anterior.
schema = {
    "type": "record",
    "name": "Diccionario_Tabla_Accidentes_2024",
    "namespace": "es.pfg.bigdata",
    "fields": fields
}

schema = parse_schema(schema)


# to_dict convierte los datos en una lista de diccionarios.
records = df.to_dict(orient="records")


# open crea el archivo de salida en modo binario de escritura ("wb").
# El archivo se denomina TABLA_ACCIDENTES_24.avro.
with open("TABLA_ACCIDENTES_24.avro", "wb") as out:
    # Escribir los registros utilizando el esquema definido anteriormente.
    # Avro almacena los datos en formato binario junto con su estructura de campos.
    writer(out, schema, records)


print("Conversión a AVRO finalizada correctamente.")

Conversión a AVRO finalizada correctamente.
